## Importar as bibliotecas

In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

In [ ]:
df = pd.read_csv("../dados/mecaniqa_dataset.csv")
df["Data"] = pd.to_datetime(df["Data"])
df = df.set_index("Data")
df = df.sort_index()

df.head()

,Trocas_Oleo,Manutencao_Motor
Data,,
2024-01-01,11.0,2.0
2024-01-02,9.0,4.0
2024-01-03,12.0,1.0
2024-01-04,15.0,8.0
2024-01-05,25.0,10.0


## Preparar as informações

In [ ]:
dados = df.sort_index().copy()
dados["Lag_1"] = dados["Trocas_Oleo"].shift(1)
dados["Lag_7"] = dados["Trocas_Oleo"].shift(7)
dados["Media_7"] = dados["Trocas_Oleo"].shift(1).rolling(7).mean()
dados["Dia_Semana"] = dados.index.dayofweek
colunas = ["Lag_1", "Lag_7", "Media_7", "Dia_Semana"]
dados = dados.dropna(subset=colunas + ["Trocas_Oleo"])

X = dados[colunas]
y = dados["Trocas_Oleo"]

## Criar os modelos

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

modelos = {
    "Árvore de Decisão": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42)
}

## Treinar e Avaliar

In [ ]:
resultados = []

for divisao, (treino, teste) in enumerate(tscv.split(X), start=1):

    X_treino = X.iloc[treino]
    X_teste = X.iloc[teste]

    y_treino = y.iloc[treino]
    y_teste = y.iloc[teste]

    previsoes = {
        "Baseline Média 7": dados["Media_7"].iloc[teste].to_numpy()
    }

    for nome, modelo in modelos.items():

        modelo.fit(X_treino, y_treino)
        previsoes[nome] = modelo.predict(X_teste)

    dados_validos = y_teste != 0

    for nome, previsao in previsoes.items():

        rmse = np.sqrt(mean_squared_error(y_teste, previsao))

        if dados_validos.any():
            mape = mean_absolute_percentage_error(
                y_teste[dados_validos],
                previsao[dados_validos.to_numpy()]
            ) * 100
        else:
            mape = np.nan

        resultados.append({
            "Modelo": nome,
            "Divisão": divisao,
            "RMSE": rmse,
            "MAPE (%)": mape
        })

## Resultados

In [ ]:
tabela_resultados = pd.DataFrame(resultados)
resumo = tabela_resultados.groupby("Modelo")[["RMSE", "MAPE (%)"]].mean()
resumo = resumo.sort_values("RMSE")

print(resumo.round(2))

                   RMSE  MAPE (%)
Modelo                           
Random Forest      6.48     24.67
Árvore de Decisão  6.57     24.23
Baseline Média 7   8.10     37.39
